# Model Training Notebook

This notebook trains a Random Forest model on the gut survey data, builds a preprocessing pipeline, and saves the trained model.

In [80]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
import joblib
import numpy as np
import re
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

## 1. Load and Inspect Data

In [81]:
# 1.1 Load your collected data
df = pd.read_csv("synthetic_bowel_movements.csv")
df.head()

,What is your age?,How would you rate the typical smell intensity of your stool?,"On average, how many hours of sleep do you get per night?",Timestamp,What is your gender?,Height (cm),Weight (kg),Hydration Level,How active are you physically on average?,Are you currently taking any medication that affects digestion or bowel movements?,...,How many meals with greasy or fried food do you eat per week?,"Do you regularly consume dairy products (milk, cheese, yogurt)?","On average, how many servings of processed food do you eat per day?","On average, how many servings of fruits and vegetables do you eat per day?",What type of toilet paper or wiping method do you usually use?,How would you describe your typical stool consistency?,What is the most common color of your stool?,"How many times do you go to the toilet for the number ""2"" in a week?","How many caffeinated beverages (coffee, tea, energy drinks) do you consume per day?","On average, how many wipes or sheets of toilet paper do you use per bowel movement?"
0,29,4,7,02/05/2025 22:48:43,Male,180,95,Moderate (1–2 liters/day),High (exercise 5+ days a week or physical job),No,...,0-2,Yes,0-2,0-2,2-ply paper,Firm and smooth,Brown,5,3-5,8
1,13,4,8,02/05/2025 23:02:38,Male,178,69,Moderate (1–2 liters/day),Low (mostly sedentary),No,...,0-2,Yes,0-2,0-2,3-ply paper,Soft,Brown,5-7,5 and more,24
2,20,2,6,02/05/2025 22:51:02,Male,161,74,Moderate (1–2 liters/day),Moderate (light exercise a few days a week),No,...,5 and more,Yes,0-2,0-2,2-ply paper,Firm and smooth,Brown,2-3,0-2,7
3,29,2,8,02/05/2025 23:02:48,Male,163,70,Moderate (1–2 liters/day),Moderate (light exercise a few days a week),No,...,0-2,Yes,3-4,0-2,3-ply paper,Firm and smooth,Brown,14,0-2,Like 10 wipes
4,18,1,4,02/05/2025 22:50:20,Male,181,69,Moderate (1–2 liters/day),High (exercise 5+ days a week or physical job),No,...,0-2,No,0-2,0-2,2-ply paper,Hard and lumpy,Brown,5,5 and more,3


In [82]:
column_mapping = {
    "What is your age?": "age",
    "How would you rate the typical smell intensity of your stool?": "smell_intensity",
    "On average, how many hours of sleep do you get per night?": "sleep_hours",
    "Timestamp": "timestamp",
    "What is your gender?": "gender",
    "Height (cm)": "height_cm",
    "Weight (kg)": "weight_kg",
    "Hydration Level": "hydration_level",
    "How active are you physically on average?": "activity_level",
    "Are you currently taking any medication that affects digestion or bowel movements?": "meds_affecting_gut",
    "How much dietary fibers do you eat daily?": "fiber_grams",
    "How much fat do you consume daily?": "fat_grams",
    "How spicy is your typical diet?": "spiciness",
    " How many meals with greasy or fried food do you eat per week?": "weekly_greasy_meals",
    "Do you regularly consume dairy products (milk, cheese, yogurt)?": "dairy_freq",
    "On average, how many servings of processed food do you eat per day?": "processed_servings",
    "On average, how many servings of fruits and vegetables do you eat per day?": "fv_servings",
    "What type of toilet paper or wiping method do you usually use?": "toilet_method",
    "How would you describe your typical stool consistency?": "stool_consistency",
    "What is the most common color of your stool?": "stool_color",
    "How many times do you go to the toilet for the number \"2\" in a week?": "weekly_bms",
    "How many caffeinated beverages (coffee, tea, energy drinks) do you consume per day?": "caffeinated_beverages_per_day",
    "On average, how many wipes or sheets of toilet paper do you use per bowel movement?": "wipes_per_bm"
}

   

   
# Apply the mapping
df.rename(columns=column_mapping, inplace=True)

In [83]:
df.head()

,age,smell_intensity,sleep_hours,timestamp,gender,height_cm,weight_kg,hydration_level,activity_level,meds_affecting_gut,...,weekly_greasy_meals,dairy_freq,processed_servings,fv_servings,toilet_method,stool_consistency,stool_color,weekly_bms,caffeinated_beverages_per_day,wipes_per_bm
0,29,4,7,02/05/2025 22:48:43,Male,180,95,Moderate (1–2 liters/day),High (exercise 5+ days a week or physical job),No,...,0-2,Yes,0-2,0-2,2-ply paper,Firm and smooth,Brown,5,3-5,8
1,13,4,8,02/05/2025 23:02:38,Male,178,69,Moderate (1–2 liters/day),Low (mostly sedentary),No,...,0-2,Yes,0-2,0-2,3-ply paper,Soft,Brown,5-7,5 and more,24
2,20,2,6,02/05/2025 22:51:02,Male,161,74,Moderate (1–2 liters/day),Moderate (light exercise a few days a week),No,...,5 and more,Yes,0-2,0-2,2-ply paper,Firm and smooth,Brown,2-3,0-2,7
3,29,2,8,02/05/2025 23:02:48,Male,163,70,Moderate (1–2 liters/day),Moderate (light exercise a few days a week),No,...,0-2,Yes,3-4,0-2,3-ply paper,Firm and smooth,Brown,14,0-2,Like 10 wipes
4,18,1,4,02/05/2025 22:50:20,Male,181,69,Moderate (1–2 liters/day),High (exercise 5+ days a week or physical job),No,...,0-2,No,0-2,0-2,2-ply paper,Hard and lumpy,Brown,5,5 and more,3


## Data cleaning

In [84]:
# 4. Clean numeric columns
numeric_cols = ['age', 'height_cm', 'weight_kg', 'sleep_hours']
for col in numeric_cols:
    # strip non‐digits (e.g. '90kg = bulking' → '90')
    df[col] = df[col].astype(str).str.extract(r'(\d+\.?\d*)')[0].astype(float)


# 6. Helper to parse ranges like "3-4", "5 and more", "0-2"
def parse_range(val):
    if pd.isna(val): 
        return np.nan
    s = str(val)
    if 'and more' in s:
        return float(re.search(r'(\d+)', s).group(1))
    m = re.match(r'(\d+)-(\d+)', s)
    if m:
        return (float(m.group(1)) + float(m.group(2))) / 2
    # fallback: extract single number
    m2 = re.search(r'(\d+)', s)
    return float(m2.group(1)) if m2 else np.nan

# 7. Apply range parser to these columns
range_cols = [
    'smell_intensity', 'weekly_greasy_meals', 'processed_servings', 
    'fv_servings', 'weekly_bms', 'wipes_per_bm'
]
for col in range_cols:
    df[col] = df[col].apply(parse_range)

# 1. Compute the mean of weekly bms excluding 100
mean_bms = df.loc[df['weekly_bms'] != 100, 'weekly_bms'].mean()
df.loc[df['weekly_bms'] == 100, 'weekly_bms'] = mean_bms

# Map Male→0, Female→1, leave Other as NaN for now
df['gender_encoded'] = df['gender'].map({'Male': 0, 'Female': 1})

# Randomly assign 0 or 1 for the rows where gender == 'Other'
mask_other = df['gender'] == 'Other'
df.loc[mask_other, 'gender_encoded'] = np.random.randint(0, 2, size=mask_other.sum())

# Drop the original gender column if you no longer need it
df.drop(['gender', 'timestamp', 'meds_affecting_gut'], axis=1, inplace=True)

# 2. One-hot encode the remaining categorical columns
categorical_cols = [
    'hydration_level',
    'activity_level',
    'dairy_freq',
    'spiciness',
    'toilet_method',
    'stool_consistency',
    'stool_color',
    'caffeinated_beverages_per_day',
    'fat_grams',
    'fiber_grams'
]

df = pd.get_dummies(df, columns=categorical_cols, drop_first=False)


# 9. Final sanity check
print(df.dtypes)
df.head()

age                                                              float64
smell_intensity                                                  float64
sleep_hours                                                      float64
height_cm                                                        float64
weight_kg                                                        float64
weekly_greasy_meals                                              float64
processed_servings                                               float64
fv_servings                                                      float64
weekly_bms                                                       float64
wipes_per_bm                                                     float64
gender_encoded                                                   float64
hydration_level_Excellent (3+ liters/day)                           bool
hydration_level_Good (2–3 liters/day)                               bool
hydration_level_Moderate (1–2 liters/day)          

,age,smell_intensity,sleep_hours,height_cm,weight_kg,weekly_greasy_meals,processed_servings,fv_servings,weekly_bms,wipes_per_bm,...,stool_color_Red or bloody,caffeinated_beverages_per_day_0-2,caffeinated_beverages_per_day_3-5,caffeinated_beverages_per_day_5 and more,fat_grams_A lot,fat_grams_Moderate,fat_grams_Not much,fiber_grams_A lot,fiber_grams_Moderate,fiber_grams_Not much
0,29.0,4.0,7.0,180.0,95.0,1.0,1.0,1.0,5.0,8.0,...,False,False,True,False,False,True,False,False,True,False
1,13.0,4.0,8.0,178.0,69.0,1.0,1.0,1.0,6.0,24.0,...,False,False,False,True,False,True,False,True,False,False
2,20.0,2.0,6.0,161.0,74.0,5.0,1.0,1.0,2.5,7.0,...,False,True,False,False,False,True,False,False,True,False
3,29.0,2.0,8.0,163.0,70.0,1.0,3.5,1.0,14.0,10.0,...,False,True,False,False,False,False,True,False,False,True
4,18.0,1.0,4.0,181.0,69.0,1.0,1.0,1.0,5.0,3.0,...,False,False,False,True,False,True,False,True,False,False


### Filling NaN

In [85]:
# 1. Compute means (excluding NaNs)
mean_weekly_bms = df['weekly_bms'].mean()
mean_wipes_per_bm = df['wipes_per_bm'].mean()  # or 'wipes_per_bm' if that's your column name

# 2. Fill NaNs with the computed means
df['weekly_bms'].fillna(mean_weekly_bms, inplace=True)
df['wipes_per_bm'].fillna(mean_wipes_per_bm, inplace=True)

/var/folders/l8/x_d5tjhx5m722v_3w4t33www0000gn/T/ipykernel_67410/2091718923.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['weekly_bms'].fillna(mean_weekly_bms, inplace=True)
/var/folders/l8/x_d5tjhx5m722v_3w4t33www0000gn/T/ipykernel_67410/2091718923.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting value

## 2. Split Features and Target

In [86]:
# 1.2 Split into X/y
X = df.drop("wipes_per_bm", axis=1)  # e.g. 'next_bowel_movement_in_hours'
y = df["wipes_per_bm"]
X.head(), y.head()

(    age  smell_intensity  sleep_hours  height_cm  weight_kg  \
 0  29.0              4.0          7.0      180.0       95.0   
 1  13.0              4.0          8.0      178.0       69.0   
 2  20.0              2.0          6.0      161.0       74.0   
 3  29.0              2.0          8.0      163.0       70.0   
 4  18.0              1.0          4.0      181.0       69.0   
 
    weekly_greasy_meals  processed_servings  fv_servings  weekly_bms  \
 0                  1.0                 1.0          1.0         5.0   
 1                  1.0                 1.0          1.0         6.0   
 2                  5.0                 1.0          1.0         2.5   
 3                  1.0                 3.5          1.0        14.0   
 4                  1.0                 1.0          1.0         5.0   
 
    gender_encoded  ...  stool_color_Red or bloody  \
 0             0.0  ...                      False   
 1             0.0  ...                      False   
 2             0.0

## 3. Build Preprocessing Pipeline

In [ ]:
# 1.3 Define numeric and categorical features using your cleaned column names
numeric_features = [
    "age",
    "height_cm",
    "weight_kg",
    "smell_intensity",
    "sleep_hours",
    "weekly_greasy_meals",
    "processed_servings",
    "fv_servings",
    "weekly_bms",
    "wipes_per_bm"
]

categorical_features = [
    "gender_encoded",
    "hydration_level",
    "activity_level",
    "dairy_freq",
    "spiciness",
    "toilet_method",
    "stool_consistency",
    "stool_color",
    "caffeinated_beverages_per_day",
    "fat_grams",
    "fiber_grams"
]

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)


## 4. Create and Train the Pipeline

In [90]:
print(df.columns)

Index(['age', 'smell_intensity', 'sleep_hours', 'height_cm', 'weight_kg',
       'weekly_greasy_meals', 'processed_servings', 'fv_servings',
       'weekly_bms', 'wipes_per_bm', 'gender_encoded',
       'hydration_level_Excellent (3+ liters/day)',
       'hydration_level_Good (2–3 liters/day)',
       'hydration_level_Moderate (1–2 liters/day)',
       'hydration_level_Poor (rarely drink water)',
       'activity_level_High (exercise 5+ days a week or physical job)',
       'activity_level_Low (mostly sedentary)',
       'activity_level_Moderate (light exercise a few days a week)',
       'dairy_freq_No', 'dairy_freq_Occasionally', 'dairy_freq_Yes',
       'spiciness_Moderate', 'spiciness_Not spicy', 'spiciness_Spicy',
       'toilet_method_1-ply paper', 'toilet_method_2-ply paper',
       'toilet_method_3-ply paper', 'toilet_method_Bidet/water',
       'toilet_method_Other', 'toilet_method_Wet wipes',
       'stool_consistency_Firm and smooth', 'stool_consistency_Hard and lumpy',
    

In [89]:
# 1.4 Create full pipeline with an estimator
pipeline = Pipeline([
    ("prep", preprocessor),
    ("clf", RandomForestClassifier(n_estimators=100, random_state=42))
])

# Train the pipeline
pipeline.fit(X, y)

ValueError: A given column is not a column of the dataframe

## 5. Save the Trained Model

In [ ]:
# 1.5 Serialize the trained pipeline
joblib.dump(pipeline, "model.pkl")
print("Model trained and saved to model.pkl")